In [18]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
    size_adjusted_power_comparison,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = True
_AUGMENTED_PARAM = 'x_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.05
_MC_SAMPLES = 1000
_MC_ALPHA = 0.05
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)



In [19]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83   0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [20]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [21]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [22]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [23]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: True
Augmented measurement equation: OutGap
Augmented coefficient: x_coef
Monte Carlo replications: 1000
Noise Covariance:
 [[0.604 0.    0.   ]
 [0.    0.839 0.   ]
 [0.    0.    0.039]]


In [24]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 1000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.495,0.421,0.062,0.009,1000,113,0.113,0.010,0.095,0.134
1,Infl,0.839,0.534,0.038,0.009,1000,40,0.040,0.006,0.030,0.054
2,Rate,1.061,0.474,0.044,0.009,1000,50,0.050,0.007,0.038,0.065


In [25]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 1000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.122,3.162,0.499,0.002,0.092,0.009,1000,68,0.068,0.008,0.054,0.085,3.0,200,4
1,cov_identity,4.184,739.146,0.000,0.016,8.432,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


In [26]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-0.259,-0.014,1.338,-0.204,0.486,0.005,0.042,0.002,0.003,0.032,0.009,0.0,1000,49,0.049,0.007,0.037,0.064
1,OutGap,x,-0.318,-0.087,0.248,-1.244,0.316,0.012,0.008,0.002,0.001,0.030,0.009,0.0,1000,223,0.223,0.013,0.198,0.250
2,OutGap,r,-1.450,-0.047,2.114,-0.664,0.438,0.007,0.069,0.002,0.008,0.032,0.010,0.0,1000,90,0.090,0.009,0.074,0.109
3,Infl,Pi,-0.551,-0.023,1.594,-0.330,0.474,0.006,0.052,0.002,0.004,0.032,0.009,0.0,1000,63,0.063,0.008,0.050,0.080
4,Infl,x,0.019,0.005,0.297,0.076,0.489,0.005,0.010,0.002,0.001,0.033,0.009,0.0,1000,61,0.061,0.008,0.048,0.078
5,Infl,r,-0.275,-0.005,2.523,-0.076,0.487,0.005,0.084,0.002,0.010,0.033,0.009,0.0,1000,57,0.057,0.007,0.044,0.073
6,Rate,Pi,-0.058,-0.014,0.303,-0.193,0.485,0.005,0.010,0.002,0.001,0.032,0.009,0.0,1000,54,0.054,0.007,0.042,0.070
7,Rate,x,0.024,0.029,0.056,0.412,0.473,0.006,0.002,0.002,0.000,0.032,0.010,0.0,1000,66,0.066,0.008,0.052,0.083
8,Rate,r,-0.049,-0.006,0.479,-0.081,0.490,0.005,0.016,0.002,0.002,0.033,0.009,0.0,1000,53,0.053,0.007,0.041,0.069


In [27]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-1.397,-0.119,0.818,-1.695,0.189,0.018,0.023,0.002,0.001,0.027,0.007,0.000,1000,355,0.355,0.015,0.326,0.385
1,OutGap,x,-0.311,-0.142,0.150,-2.024,0.113,0.023,0.005,0.002,0.000,0.026,0.005,0.001,1000,501,0.501,0.016,0.470,0.532
0,OutGap,r,-0.355,-0.012,2.029,-0.175,0.553,0.004,0.054,0.002,0.007,0.027,0.009,0.000,1000,23,0.023,0.005,0.015,0.034
5,Infl,Pi,-0.309,-0.021,0.982,-0.296,0.484,0.006,0.032,0.002,0.001,0.032,0.009,0.000,1000,59,0.059,0.007,0.046,0.075
4,Infl,x,-0.038,-0.012,0.181,-0.168,0.493,0.005,0.006,0.002,0.000,0.032,0.009,0.000,1000,60,0.060,0.008,0.047,0.076
3,Infl,r,0.041,0.002,2.418,0.030,0.498,0.005,0.078,0.002,0.009,0.033,0.009,0.000,1000,54,0.054,0.007,0.042,0.070
8,Rate,Pi,0.026,0.009,0.187,0.129,0.506,0.005,0.006,0.002,0.000,0.032,0.009,0.000,1000,45,0.045,0.007,0.034,0.060
7,Rate,x,0.013,0.024,0.034,0.339,0.481,0.006,0.001,0.002,0.000,0.032,0.009,0.000,1000,74,0.074,0.008,0.059,0.092
6,Rate,r,-0.071,-0.009,0.459,-0.121,0.497,0.005,0.015,0.002,0.002,0.032,0.009,0.000,1000,51,0.051,0.007,0.039,0.066


In [28]:
print("Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"]).round(3)

Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.759,-2.018,-0.259,-0.259,0.0,0.0,0.0,0.028,0.021,0.042,0.042,0.0,0.0,0.0
1,OutGap,x,0.003,-0.321,-0.318,-0.318,-0.0,0.0,0.0,0.005,0.004,0.008,0.008,0.0,0.0,0.0
2,OutGap,r,-0.075,-1.375,-1.450,-1.450,0.0,0.0,0.0,0.044,0.036,0.069,0.069,0.0,0.0,0.0
3,Infl,Pi,-0.039,-0.512,-0.551,-0.551,-0.0,0.0,0.0,0.012,0.051,0.052,0.052,0.0,0.0,0.0
4,Infl,x,0.004,0.016,0.019,0.019,0.0,0.0,0.0,0.002,0.010,0.010,0.010,0.0,0.0,0.0
5,Infl,r,-0.035,-0.239,-0.275,-0.275,0.0,0.0,0.0,0.019,0.084,0.084,0.084,0.0,0.0,0.0
6,Rate,Pi,0.004,-0.062,-0.058,-0.058,-0.0,0.0,0.0,0.003,0.010,0.010,0.010,0.0,0.0,0.0
7,Rate,x,-0.000,0.024,0.024,0.024,0.0,0.0,0.0,0.000,0.002,0.002,0.002,0.0,0.0,0.0
8,Rate,r,-0.011,-0.038,-0.049,-0.049,0.0,0.0,0.0,0.004,0.016,0.016,0.016,0.0,0.0,0.0


In [29]:
print("Innovation decomposition on raw predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"]).round(3)

Innovation decomposition on raw predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.843,-3.239,-1.397,-1.397,0.0,0.0,0.0,0.017,0.009,0.023,0.023,0.0,0.0,0.0
1,OutGap,x,0.267,-0.578,-0.311,-0.311,-0.0,0.0,0.0,0.003,0.003,0.005,0.005,0.0,0.0,0.0
2,OutGap,r,-0.921,0.565,-0.355,-0.355,-0.0,0.0,0.0,0.053,0.032,0.054,0.054,0.0,0.0,0.0
3,Infl,Pi,-0.016,-0.293,-0.309,-0.309,-0.0,0.0,0.0,0.007,0.031,0.032,0.032,0.0,0.0,0.0
4,Infl,x,-0.002,-0.036,-0.038,-0.038,-0.0,0.0,0.0,0.001,0.006,0.006,0.006,0.0,0.0,0.0
5,Infl,r,-0.020,0.061,0.041,0.041,-0.0,0.0,0.0,0.018,0.078,0.078,0.078,0.0,0.0,0.0
6,Rate,Pi,0.005,0.022,0.026,0.026,0.0,0.0,0.0,0.002,0.006,0.006,0.006,0.0,0.0,0.0
7,Rate,x,0.001,0.012,0.013,0.013,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.010,-0.062,-0.071,-0.071,-0.0,0.0,0.0,0.004,0.015,0.015,0.015,0.0,0.0,0.0


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.

In [30]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()



## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [31]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.317,-1334.968,-1049.061,571.815,0.0,0.002,1.669,0.495,2.64,0.0,1000,1000,1.0,0.0,0.996,1.0


In [32]:
res_mle

OptimizationResult(kind='mle', x=array([1.18874448]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(1.1887444849673674), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(1026.9796329523222), loglik=np.float64(-1026.9796329523222), logprior=np.float64(0.0), logpost=np.float64(-1026.9796329523222), nfev=14, nit=6, raw=  message: CONVERGENCE

## Serial Autocorrelation Tests for the Augmented Model

In [33]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.880,0.371,0.073,0.009,1000,152,0.152,0.011,0.131,0.176
1,Infl,4.248,0.160,0.112,0.007,1000,462,0.462,0.016,0.431,0.493
2,Rate,1.069,0.481,0.046,0.009,1000,46,0.046,0.007,0.035,0.061


In [34]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.122,3.162,0.499,0.002,0.092,0.009,1000,68,0.068,0.008,0.054,0.085,3.0,200,4
1,cov_identity,4.184,739.146,0.000,0.016,8.432,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.097,3.025,0.504,0.001,0.081,0.009,1000,52,0.052,0.007,0.040,0.068,3.0,200,4
1,cov_identity,0.564,162.727,0.000,0.002,2.323,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


,test,n_replications,distance_ref,mc_se_distance_ref,distance_aug,mc_se_distance_aug,distance_improvement,mc_se_distance_improvement,stat_ref,mc_se_stat_ref,stat_aug,mc_se_stat_aug,stat_improvement,mc_se_stat_improvement,aug_closer_rate,aug_closer_rate_mc_se,aug_closer_ci_low,aug_closer_ci_high
0,mean_zero_hac,1000,0.122,0.002,0.097,0.001,0.025,0.001,3.162,0.092,3.025,0.081,0.137,0.025,0.865,0.011,0.842,0.885
1,cov_identity,1000,4.184,0.016,0.564,0.002,3.621,0.016,739.146,8.432,162.727,2.323,576.419,7.308,1.000,0.000,0.996,1.000
